# Barttorvik Full Season

Get the Barttorvik ratings from previous full seasons (regular season plus postseason) for features about previous team strength. Assumes all data has been copied from website (https://barttorvik.com/#) to Excel, as specified in the steps below. 

Steps before this file:

1. Copy from the website, excluding the initial row with D1 averages
2. Make sure the REC column in excel is text only before pasting (otherwise it tries to convert the records into a date)
3. Paste into excel with "Match Destination Formatting"
4. Save as csv into the folder

Note: Do not copy from previous year's project because the team naming conventions sometimes switch and this file relies on consistent names across seasons (i.e. North Carolina State in 2024 to N.C. State in 2025)

In [1]:
SEASON = 2025

In [2]:
import pandas as pd

pd.set_option('display.max_columns', 100)

def load_bartorvik(season: int):
    d = pd.read_excel(f"../data/unprocessed/mens_barttorvik_full_season/barttorvik_full_season_{season}.xlsx")
    d.columns = d.columns.str.upper()

    return d

df = pd.concat(
    [
        load_bartorvik(season)
        for season in range(2008, SEASON)
    ],
    ignore_index=True
)

df.insert(0, 'Season', df.pop('YEAR'))
df['ADJ EM'] = df['ADJ OE'] - df['ADJ DE']

df = df[['Season', 'TEAM', 'BARTHAG', 'ADJ EM']]

df

,Season,TEAM,BARTHAG,ADJ EM
0,2008,Kansas,0.982,35.653
1,2008,Memphis,0.970,30.779
2,2008,UCLA,0.965,29.207
3,2008,North Carolina,0.964,30.808
4,2008,Wisconsin,0.958,26.972
...,...,...,...,...
5955,2024,VMI,0.065,-22.876
5956,2024,IU Indy,0.064,-24.441
5957,2024,Saint Francis,0.063,-24.653
5958,2024,Coppin St.,0.045,-25.961


Some teams (like Ivy League in 2021) are missing, so add them in with NAs

In [3]:
df_full_teams = pd.DataFrame(
    [(season, team) for team in df['TEAM'].unique() for season in df['Season'].unique()],
    columns=['Season', 'TEAM']
)

df_full_teams

,Season,TEAM
0,2008,Kansas
1,2009,Kansas
2,2010,Kansas
3,2011,Kansas
4,2012,Kansas
...,...,...
6234,2020,Le Moyne
6235,2021,Le Moyne
6236,2022,Le Moyne
6237,2023,Le Moyne


In [4]:
df = (
    pd.merge(
        df, 
        df_full_teams,
        how='right',
        on=['Season', 'TEAM']
    )
    .sort_values(
        ['Season', 'TEAM'], 
        ignore_index=True
    )
)

df

,Season,TEAM,BARTHAG,ADJ EM
0,2008,Abilene Christian,NaN,NaN
1,2008,Air Force,0.563,2.163
2,2008,Akron,0.751,10.037
3,2008,Alabama,0.755,10.303
4,2008,Alabama A&M,0.127,-16.024
...,...,...,...,...
6234,2024,Wright St.,0.553,2.124
6235,2024,Wyoming,0.530,1.125
6236,2024,Xavier,0.800,12.669
6237,2024,Yale,0.723,8.782


In [5]:
df['Past 4 Years BARTHAG'] = (
    df
    .groupby(['TEAM'])
    ['BARTHAG']
    .rolling(window=4, min_periods=2)  # at least 2 years of data to calculate
    .mean()
    .reset_index()
    .set_index('level_1')
)['BARTHAG']

df['Past 4 Years ADJ EM'] = (
    df
    .groupby(['TEAM'])
    ['ADJ EM']
    .rolling(window=4, min_periods=2)  # at least 2 years of data to calculate
    .mean()
    .reset_index()
    .set_index('level_1')
)['ADJ EM']

df.rename(
    columns={
        'BARTHAG': 'Past Year BARTHAG',
        'ADJ EM': 'Past Year ADJ EM'
    }, 
    inplace=True
)

df['Season'] += 1  # shift by a year so metrics are from past instead of the current rating

df = df.loc[df['Season'] >= 2012, :].reset_index(drop=True)

df

,Season,TEAM,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
0,2012,Abilene Christian,NaN,NaN,NaN,NaN
1,2012,Air Force,0.578,2.881,0.462750,-1.377750
2,2012,Akron,0.605,3.741,0.677500,6.696000
3,2012,Alabama,0.842,14.255,0.780750,11.435000
4,2012,Alabama A&M,0.128,-16.046,0.103500,-18.409250
...,...,...,...,...,...,...
5133,2025,Wright St.,0.553,2.124,0.551750,2.084500
5134,2025,Wyoming,0.530,1.125,0.590750,3.550500
5135,2025,Xavier,0.800,12.669,0.825750,14.518500
5136,2025,Yale,0.723,8.782,0.673667,6.763333


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5138 entries, 0 to 5137
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Season                5138 non-null   int64  
 1   TEAM                  5138 non-null   object 
 2   Past Year BARTHAG     4928 non-null   float64
 3   Past Year ADJ EM      4928 non-null   float64
 4   Past 4 Years BARTHAG  4928 non-null   float64
 5   Past 4 Years ADJ EM   4928 non-null   float64
dtypes: float64(4), int64(1), object(1)
memory usage: 241.0+ KB


In [8]:
df.to_parquet(f'../data/preprocessed/mens_barttorvik_full_season/barttorvik_full_season.parquet')

'Done'

'Done'